### 4.HMM实践

我们使用hmmlearn实践一下。详细参见[官方文档](https://hmmlearn.readthedocs.io/en/latest/)：https://hmmlearn.readthedocs.io/en/latest/

hmmlearn实现了三种HMM模型类，按照观测状态是连续状态还是离散状态，可以分为两类。GaussianHMM和GMMHMM是连续观测状态的HMM模型，而MultinomialHMM是离散观测状态的模型，也是我们在HMM原理系列篇里面使用的模型。

以下是使用HMM进行采样的例子

In [ ]:
# --- 隐马尔可夫模型（HMM）实践 ---
# HMM 由三组参数定义：初始状态概率(pi)、状态转移矩阵(A)、观测概率/发射概率(B)
# 本例使用连续观测的高斯HMM：每个隐状态对应一个高斯分布 N(mu, Sigma)
import numpy as np
import matplotlib.pyplot as plt
from hmmlearn import hmm

# 定义4个隐状态的 HMM 参数
# 初始状态概率：模型在 t=0 时刻处于各状态的概率
startprob = np.array([0.6, 0.3, 0.1, 0.0])
# 状态转移矩阵 A[i][j] = P(下一状态=j | 当前状态=i)
# 注意：状态1和状态3之间无直接转移（A[0][2]=A[2][0]=0）
transmat = np.array([[0.7, 0.2, 0.0, 0.1],
                     [0.3, 0.5, 0.2, 0.0],
                     [0.0, 0.3, 0.5, 0.2],
                     [0.2, 0.0, 0.2, 0.6]])
# 每个隐状态对应的高斯分布均值（二维观测）
means = np.array([[0.0, 0.0],
                  [0.0, 11.0],
                  [9.0, 10.0],
                  [11.0, -1.0]])
# 每个隐状态的协方差矩阵（此处为单位矩阵的0.5倍，即各向同性高斯）
covars = .5 * np.tile(np.identity(2), (4, 1, 1))

# 构建高斯 HMM 实例
gen_model = hmm.GaussianHMM(n_components=4, covariance_type="full")
# 直接设置模型参数（而非从数据中学习）
gen_model.startprob_ = startprob
gen_model.transmat_ = transmat
gen_model.means_ = means
gen_model.covars_ = covars

# 从 HMM 中采样：生成500个观测点及其对应的真实隐状态序列
# X 为观测序列（500x2），Z 为隐状态序列（长度500）
X, Z = gen_model.sample(500)

# 可视化采样数据：观测轨迹在不同状态的高斯分布之间跳跃
fig, ax = plt.subplots()
ax.plot(X[:, 0], X[:, 1], ".-", label="observations", ms=6,
        mfc="orange", alpha=0.7)
# 标注每个隐状态的均值位置
for i, m in enumerate(means):
    ax.text(m[0], m[1], 'Component %i' % (i + 1),
            size=17, horizontalalignment='center',
            bbox=dict(alpha=.7, facecolor='w'))
ax.legend(loc='best')
fig.show()

In [ ]:
# --- 模型选择与训练 ---
# 尝试不同隐状态数量（3, 4, 5），通过在验证集上的对数似然分数选择最优模型
# 这是 HMM 模型选择的标准方法
scores = list()
models = list()
for n_components in (3, 4, 5):
    # 定义高斯 HMM，n_iter=10 为 Baum-Welch 算法的最大迭代次数
    # Baum-Welch 算法是 EM 算法在 HMM 上的应用：E步计算期望，M步更新参数
    model = hmm.GaussianHMM(n_components=n_components,
                            covariance_type='full', n_iter=10)
    model.fit(X[:X.shape[0] // 2])  # 前半部分作为训练集
    models.append(model)
    # score 计算后半部分数据的对数似然（越大越好），用于模型评估
    scores.append(model.score(X[X.shape[0] // 2:]))
    print(f'Converged: {model.monitor_.converged}'
          f'\tScore: {scores[-1]}')

# 选择验证集上得分最高的模型
model = models[np.argmax(scores)]
n_states = model.n_components
print(f'The best model had a score of {max(scores)} and {n_states} '
      'states')

# 使用 Viterbi 算法解码：给定观测序列，找出最可能的隐状态序列
# Viterbi 是一种动态规划算法，时间复杂度 O(T * K^2)，T为序列长度，K为状态数
states = model.predict(X)

In [ ]:
# --- 模型评估：比较生成的状态与恢复的状态 ---
# 将模型预测的状态序列与真实生成的状态序列进行对比
fig, ax = plt.subplots()
ax.plot(Z, states)  # 如果模型恢复得好，应接近对角线
ax.set_title('States compared to generated')
ax.set_xlabel('Generated State')
ax.set_ylabel('Recovered State')
fig.show()

# 比较真实的转移矩阵与模型学到的转移矩阵
# 色彩越接近说明模型恢复的参数越准确
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 5))
ax1.imshow(gen_model.transmat_, aspect='auto', cmap='spring')
ax1.set_title('Generated Transition Matrix')
ax2.imshow(model.transmat_, aspect='auto', cmap='spring')
ax2.set_title('Recovered Transition Matrix')
for ax in (ax1, ax2):
    ax.set_xlabel('State To')
    ax.set_ylabel('State From')
fig.tight_layout()
fig.show()